# COMPARISON PANDAS VS DUCKDB

In [ ]:
# ============================================================
# Pandas vs DuckDB CSV Performance Benchmark
# Author: Ghazian Hanafi
# Purpose:
#   Compare memory usage and execution time between Pandas
#   and DuckDB when opening and exploring CSV files.
# Data: IMDB Movie Dataset (380 MB) and OWID CO2 Dataset (13 MB)
# Data sources:
#   - IMDB Movie Dataset:
#   - OWID CO2 Dataset:  
# Datasets:
#   - Medium (~13 MB)
#   - Large (~500 MB)
# ============================================================

import pandas as pd
import duckdb
import os
import time
import psutil

# ============================================================
# CONFIGURATION
# ============================================================
large_dir = "C:\\Users\\ghazi\\OneDrive\\Bismillah Intellectual Property of Ghazian Hirzi Hanafi\\SCI035_RESEARCH\\08_Portofolio\\11_IMDB_dataset\\Data\\"
MEDIUM_CSV = "global_owid-co2-data.csv"   # ~13 MB
LARGE_CSV  = large_dir+"Imdb Movie Dataset.csv"   # replace with your actual file

process = psutil.Process(os.getpid())

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def memory_mb():
    """Return current process memory usage in MB"""
    return process.memory_info().rss / (1024 * 1024)

def benchmark(func, label):
    """Measure execution time and memory usage"""
    mem_before = memory_mb()
    start = time.time()

    result = func()

    elapsed = time.time() - start
    mem_after = memory_mb()

    print("=" * 60)
    print(label)
    print(f"Time elapsed : {elapsed:.2f} seconds")
    print(f"Memory delta : {mem_after - mem_before:.2f} MB")
    print("=" * 60, "\n")

    return result

# ============================================================
# MEDIUM DATASET (~13 MB)
# ============================================================

print("\n###============== MEDIUM DATASET (~13 MB) ==============###\n")

# Pandas: open CSV
df_pandas_medium = benchmark(
    lambda: pd.read_csv(MEDIUM_CSV),
    "Pandas – Open Medium CSV"
)
duckdb.sql()
# DuckDB: open CSV (lazy scan)
df_duckdb_medium = benchmark(
    lambda: duckdb.sql(f"""
        SELECT *
        FROM read_csv_auto('{MEDIUM_CSV}')
        LIMIT 10
    """).df(),
    "DuckDB – Open Medium CSV"
)

# Exploration comparison
# benchmark(
#     lambda: df_pandas_medium.describe(),
#     "Pandas – describe() (Medium)"
# )

# benchmark(
#     lambda: duckdb.sql(f"""
#         SELECT
#             COUNT(*) AS rows,
#             AVG(co2_per_capita) AS avg_co2_pc
#         FROM read_csv_auto('{MEDIUM_CSV}')
#     """).df(),
#     "DuckDB – aggregation (Medium)"
# )

# ============================================================
# LARGE DATASET (~380 MB)
# ============================================================

print("\n###============== LARGE DATASET (~380 MB) ==============###\n")

# Pandas: open CSV (WARNING: high RAM usage)
df_pandas_large = benchmark(
    lambda: pd.read_csv(LARGE_CSV),
    "Pandas – Open Large CSV"
)

# DuckDB: open CSV (safe, lazy)
df_duckdb_large = benchmark(
    lambda: duckdb.sql(f"""
        SELECT *
        FROM read_csv_auto('{LARGE_CSV}')
        LIMIT 10
    """).df(),
    "DuckDB – Open Large CSV"
)

# Aggregation comparison
# benchmark(
#     lambda: duckdb.sql(f"""
#         SELECT
#             COUNT(*) AS rows,
#             AVG(revenue) AS avg_revenue
#         FROM read_csv_auto('{LARGE_CSV}')
#     """).df(),
#     "DuckDB – aggregation (Large)"
# )

# benchmark(
#     lambda: df_pandas_large["revenue"].mean(),
#     "Pandas – aggregation (Large)"
# )

print("\nBenchmark completed.\n")



###============== MEDIUM DATASET (~13 MB) ==============###

Pandas – Open Medium CSV
Time elapsed : 0.19 seconds
Memory delta : 32.81 MB

DuckDB – Open Medium CSV
Time elapsed : 0.46 seconds
Memory delta : 3.50 MB


###============== LARGE DATASET (~380 MB) ==============###

Pandas – Open Large CSV
Time elapsed : 5.90 seconds
Memory delta : 726.02 MB

DuckDB – Open Large CSV
Time elapsed : 0.24 seconds
Memory delta : 4.36 MB


Benchmark completed.



In [19]:
duckdb.sql("""
SELECT *
FROM read_csv_auto('global_owid-co2-data.csv')
LIMIT 0;
""").show()


┌─────────┬───────┬──────────┬────────────┬────────┬────────────┬───────────────────────┬────────┬────────────────┬─────────────────┬───────────────────┬──────────────────────────────┬───────────────────────────────┬──────────────────────────────┬───────────────────────────┬───────────────────────────────────┬────────────────┬─────────────┬─────────────────────┬──────────┬─────────────────────┬─────────────────┬────────────────────────────┬─────────────────────────┬───────────────────────┬────────────────┬──────────────────────────────┬─────────────────────┬────────────────────────┬────────────────────┬────────────────────┬────────────────────┬──────────────────────┬───────────────────┬────────────────┬─────────────┬────────────────────────┬─────────┬────────────────────┬───────────────────────────────┬────────────────┬─────────────────────┬────────────────────────────────┬─────────┬────────────────────┬───────────────┬──────────────────────────┬─────────┬────────────────────┬─────────

# DATA OBFUSCATION

Paper source: 
1. [Data Obfuscation: A New Class of Security 
Mechanism Providing Anonymity and Desensitization of Useable Data Sets ](https://eecs.wsu.edu/~bakken/tenure/data_obfuscation_submitted_IEEE_Privacy.pdf)
2. [Data Obfuscation: Anonymity
and Desensitization of 
Usable Data Sets](https://blough.ece.gatech.edu/research/papers/ieeesp04.pdf)


<img src=image.png width="450">



In [39]:
import duckdb
duckdb.sql(""" 
           SELECT  year,co2_per_capita 
           FROM read_csv_auto('global_owid-co2-data.csv') 
           WHERE 
           year >1990 and
           country = 'Indonesia' 
           ORDER BY year DESC
           LIMIT 10
           """).show()

┌───────┬────────────────┐
│ year  │ co2_per_capita │
│ int64 │     double     │
├───────┼────────────────┤
│  2023 │          2.608 │
│  2022 │          2.643 │
│  2021 │          2.239 │
│  2020 │          2.213 │
│  2019 │          2.399 │
│  2018 │          2.201 │
│  2017 │          2.083 │
│  2016 │          2.041 │
│  2015 │          2.059 │
│  2014 │          1.885 │
├───────┴────────────────┤
│ 10 rows      2 columns │
└────────────────────────┘



In [27]:
duckdb.sql("""
    SELECT
        hash(country) AS country_token,
        co2_per_capita
    FROM read_csv_auto('global_owid-co2-data.csv')
    WHERE year > 1990
      AND country = 'Indonesia'
""").show()


┌────────────────────┬────────────────┐
│   country_token    │ co2_per_capita │
│       uint64       │     double     │
├────────────────────┼────────────────┤
│ 978259134559100347 │          0.936 │
│ 978259134559100347 │          1.049 │
│ 978259134559100347 │          1.113 │
│ 978259134559100347 │          1.116 │
│ 978259134559100347 │          1.113 │
│ 978259134559100347 │          1.245 │
│ 978259134559100347 │          1.355 │
│ 978259134559100347 │          1.165 │
│ 978259134559100347 │          1.371 │
│ 978259134559100347 │          1.302 │
│          ·         │            ·   │
│          ·         │            ·   │
│          ·         │            ·   │
│ 978259134559100347 │          1.885 │
│ 978259134559100347 │          2.059 │
│ 978259134559100347 │          2.041 │
│ 978259134559100347 │          2.083 │
│ 978259134559100347 │          2.201 │
│ 978259134559100347 │          2.399 │
│ 978259134559100347 │          2.213 │
│ 978259134559100347 │          2.239 │


In [37]:
duckdb.sql("""
    SELECT
        year,
        round(co2_per_capita * (1 + (hash(year) % 10) / 100.0), 2) AS co2_per_capita
    FROM read_csv_auto('global_owid-co2-data.csv')
    WHERE year > 1990
      AND country = 'Indonesia'
""").show()


┌───────┬────────────────┐
│ year  │ co2_per_capita │
│ int64 │     double     │
├───────┼────────────────┤
│  1991 │           1.01 │
│  1992 │           1.05 │
│  1993 │           1.19 │
│  1994 │           1.17 │
│  1995 │           1.18 │
│  1996 │           1.36 │
│  1997 │           1.48 │
│  1998 │           1.22 │
│  1999 │           1.48 │
│  2000 │           1.38 │
│    ·  │             ·  │
│    ·  │             ·  │
│    ·  │             ·  │
│  2014 │           2.05 │
│  2015 │           2.12 │
│  2016 │           2.16 │
│  2017 │           2.27 │
│  2018 │           2.22 │
│  2019 │           2.49 │
│  2020 │           2.21 │
│  2021 │           2.35 │
│  2022 │           2.64 │
│  2023 │           2.76 │
├───────┴────────────────┤
│ 33 rows      2 columns │
│ (20 shown)             │
└────────────────────────┘

